# 🚀 AI Multi-Docs Extraction Pipeline: Step-by-Step Walkthrough (Per-Batch)

สมุดบันทึก (Jupyter Notebook) สำหรับทดสอบกระบวนการสกัดและประมวลผลเอกสาร **ทีละขั้นตอน (Step-by-Step Execution & Observability)** ในระดับ **`batch_id`**
- 📂 **ชุดข้อมูลทดสอบ**: สกัด 3 หน้าแรกจาก `Grab_202606_000007.pdf` เป็นไฟล์ `Grab_Sample_3Pages.pdf` จัดเก็บไว้ใต้ `01_drop_zone/Test_Walkthrough/`
- 🛡️ **ระบบ Per-Batch Isolation**: ทุกขั้นตอน (Stage 3 ➔ 4 ➔ 5) กำหนดให้ใช้ `batch_id` เป็นแกนหลัก เพื่อป้องกัน Process ชนกัน 100%

## 🛠️ Step 0: ตั้งค่า Working Directory, นำเข้าโมดูล และเตรียมไฟล์ทดสอบ

In [ ]:
import os
import sys
import glob
import json
import shutil
import pandas as pd
from IPython.display import display, JSON
from dotenv import load_dotenv

# 1. Ensure working directory is set to project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"📂 Project Root Working Directory: {os.getcwd()}")

# 2. Load Environment Variables & Pipeline Services
load_dotenv()
from src.application.pipeline import (
    init_system,
    split_and_match,
    extract_documents,
    async_extract_documents,
    validate_documents,
    transform_to_db,
    reset_pipeline_data,
    release_pending_merchant_files,
)
from src.infrastructure.common.config_loader import load_system_settings, get_default_doc_type, get_default_company_code
from src.infrastructure.storage.storage_manager import storage_manager
from src.infrastructure.persistence import (
    get_pending_merchants,
    approve_merchant,
    get_db_session,
    ExpenseReceipt,
    ExpenseReceiptItem,
)
from sqlalchemy import select, func

DOC_TYPE = get_default_doc_type()
COMPANY_CODE = get_default_company_code()
ACTIVE_BATCH_ID = None
print(f"✅ Pipeline Services Ready. Active Doc Type: '{DOC_TYPE}' | Company: '{COMPANY_CODE}'")

# 3. Ensure Test Fixture in Test_Walkthrough drop zone
walkthrough_drop_dir = storage_manager.get_drop_zone_dir(COMPANY_CODE, DOC_TYPE, "Test_Walkthrough")
os.makedirs(walkthrough_drop_dir, exist_ok=True)
test_pdf_target = os.path.join(walkthrough_drop_dir, "Grab_Sample_3Pages.pdf").replace("\\", "/")
master_fixture = "tests/fixtures/sample_docs/Grab_Sample_3Pages.pdf"

if not os.path.exists(test_pdf_target) and os.path.exists(master_fixture):
    shutil.copy(master_fixture, test_pdf_target)
    print(f"📋 Copied sample test file to Drop Zone: {test_pdf_target}")
else:
    print(f"📋 Test File Ready at: {test_pdf_target}")

## 🧹 Step 0.1: (Optional) รีเซ็ตระบบจาก 0 (Drop Database & Clean Fresh Start)
กดรันเซลล์นี้เมื่อต้องการ **Drop Database SQLite ทิ้งแล้ว Re-seed Master Data ใหม่จาก 0** พร้อมทั้ง **ล้างไฟล์ชั่วคราวใน `03_preprocess/` และ `04_processing/`** และเตรียมไฟล์ตัวอย่างใหม่ใน `Test_Walkthrough/` เพื่อเริ่มทดสอบใหม่ตั้งแต่ต้น

In [ ]:
# รีเซ็ต Pipeline Storage Temp และ Database สำหรับรอบการทดสอบใหม่ (เริ่มจาก 0)
reset_result = reset_pipeline_data(doc_type=DOC_TYPE, clear_storage_temp=True, clear_database=True)
print("🧹 Reset Pipeline Data Result:", reset_result)

# นำเข้าไฟล์ตัวอย่าง Grab 3 หน้าใหม่หากยังไม่มีใน Drop Zone
if os.path.exists(master_fixture) and not os.path.exists(test_pdf_target):
    shutil.copy(master_fixture, test_pdf_target)
    print(f"🔄 Restored Test Fixture: {test_pdf_target}")

print("🎉 Ready for a brand new clean step-by-step walkthrough run!")

## ⚙️ Step 1: System Initialization (`Run_01`)
ตรวจสอบความพร้อมของไฟล์คอนฟิก `settings.json`, Schema ฐานข้อมูล SQLite, และโครงสร้างโฟลเดอร์ใน `storage/`

In [ ]:
print("--- [Stage 1] Initializing System & Validating Environment ---")
init_success = init_system(drop_and_recreate=False)
if init_success:
    print("🎉 System is READY and all storage folders & DB tables are verified!")
else:
    print("❌ System initialization encountered errors. Please check configs or .env")

## 📄 Step 2: Ingest, Classify & Split PDFs (`Run_02`)
อ่านไฟล์เอกสารตัวอย่าง `Grab_Sample_3Pages.pdf` จาก `01_drop_zone/Test_Walkthrough/` เพื่อ:
1. ตรวจจับร้านค้า (Merchant Matching: Fast Prefix / Ingestion Classification)
2. ตัดหน้า PDF เป็นไฟล์ภาพ `.jpg` 3 หน้าลงใน `03_preprocess/`
3. ลงทะเบียน Batch & Pages เข้าสู่ฐานข้อมูล SQLite และส่งออก `batch_id`

In [ ]:
print("--- [Stage 2] Ingesting Documents from Drop Zone ---")
split_results = split_and_match(doc_type=DOC_TYPE, input_file=test_pdf_target, company_code=COMPANY_CODE)

if split_results:
    ACTIVE_BATCH_ID = split_results[0].get("batch_id")
    print(f"\n✅ Successfully processed batch:")
    print(f"📦 ACTIVE BATCH ID: {ACTIVE_BATCH_ID}")
    print(f"   - Matched Merchant: {split_results[0].get('matched_source')}")
    print(f"   - Total Pages: {split_results[0].get('total_pages')}")
    print(f"   - Split Page Images ({len(split_results[0].get('page_images', []))} files):")
    for img in split_results[0].get('page_images', []):
        print(f"     🖼️ {img}")
else:
    print("ℹ️ File held in PENDING awaiting merchant approval. (See Step 2.1 below)")

## 🔍 Step 2.1: Check & Confirm Pending Merchants (Human-in-the-Loop)
หากเอกสารมาจากร้านค้าใหม่ที่ยังไม่อยู่ในระบบ หรือยังไม่ได้รับการอนุมัติ เอกสารจะถูกพักไว้ที่ `02_raw_data/PENDING/`
เซลล์นี้ช่วยให้สามารถตรวจสอบและอนุมัติร้านค้า (`approve_merchant`) พร้อมทั้งปล่อยเอกสาร (`release_pending_merchant_files`) เข้าสู่กระบวนการสกัดข้อมูลต่อไปได้ทันที

In [ ]:
print("--- [Stage 2.1] Checking Pending Merchants Waiting for Approval ---")
pending_list = get_pending_merchants()

if pending_list:
    print(f"🔍 Found {len(pending_list)} pending merchant(s):\n")
    for p in pending_list:
        m_id = p.get("merchant_id")
        m_name = p.get("merchant_name")
        tax_id = p.get("tax_id")
        short_name = p.get("short_name") or "grab"
        print(f"📌 Pending Merchant: {m_name} (ID: {m_id}, Tax ID: {tax_id})")
        
        approved, msg = approve_merchant(merchant_id=m_id, short_name=short_name)
        print(f"   ➔ Approval Status: {approved} ({msg if not approved else 'Success'})")
        
        released_files = release_pending_merchant_files(doc_type=DOC_TYPE, tax_id=tax_id, short_name=short_name, company_code=COMPANY_CODE)
        if released_files:
            ACTIVE_BATCH_ID = released_files[0].get("batch_id")
            print(f"   ➔ Released Batch ID: {ACTIVE_BATCH_ID} ({len(released_files)} file batch)")
else:
    print("✅ No pending merchants awaiting review.")

print(f"\n🎯 Target Active Batch ID for Stage 3-5: '{ACTIVE_BATCH_ID}'")

## 🤖 Step 3: AI Document Extraction (`Run_03`)
นำรูปภาพหน้าที่ตัดแล้วใน `03_preprocess/` ของ **`ACTIVE_BATCH_ID`** ส่งให้ Multimodal AI สกัดข้อมูลตามโครงสร้าง `extract-schema.json`
และบันทึกไฟล์ JSON ที่สกัดได้ลงใน `04_processing/` พร้อมระบบ Smart Checkpointing ป้องกันการเรียก AI ซ้ำ

In [ ]:
print(f"--- [Stage 3] Extracting Document Data for Batch: '{ACTIVE_BATCH_ID}' ---")
if not ACTIVE_BATCH_ID:
    raise ValueError("ACTIVE_BATCH_ID is not set. Please run Step 2 / Step 2.1 first.")

extract_result = extract_documents(batch_id=ACTIVE_BATCH_ID, doc_type=DOC_TYPE, company_code=COMPANY_CODE)
print(f"📊 Extraction Summary: {extract_result}")

# Preview extracted JSON payload files
queue_pattern = f"{storage_manager.get_processing_dir(COMPANY_CODE, DOC_TYPE)}/**/*.json"
queue_files = glob.glob(queue_pattern, recursive=True)
if queue_files:
    print(f"\n💾 Found {len(queue_files)} extracted JSON file(s):\n")
    sample_file = queue_files[0]
    print(f"   Showing preview from: {sample_file}")
    with open(sample_file, "r", encoding="utf-8") as jf:
        sample_json = json.load(jf)
    display(JSON(sample_json))
else:
    print("ℹ️ No JSON files found in extracted storage.")

## 🛡️ Step 4: Validate & Post-Process Data (`Run_04`)
ตรวจสอบความถูกต้องของข้อมูลจริงใน `04_processing/` สำหรับ **`ACTIVE_BATCH_ID`**:
- ตรวจสอบเลขประจำตัวผู้เสียภาษี 13 หลัก (Tax ID Validation)
- ปรับรูปแบบวันที่ปี พ.ศ. ➔ ค.ศ. (BE to AD Normalization)
- ตรวจสอบสูตรการเงิน (Subtotal - Discount + VAT == Net)
- ตรวจสอบผลรวมรายการสินค้าเทียบกับ Subtotal
- กำหนดสถานะ `PROCESSED` หรือ `NEEDS_REVIEW`

In [ ]:
print(f"--- [Stage 4] Validating Records for Batch: '{ACTIVE_BATCH_ID}' ---")
if not ACTIVE_BATCH_ID:
    raise ValueError("ACTIVE_BATCH_ID is not set. Please run Step 2 / Step 3 first.")

validate_result = validate_documents(batch_id=ACTIVE_BATCH_ID, doc_type=DOC_TYPE, company_code=COMPANY_CODE)
print(f"📊 Validation Summary: {validate_result}")

## 💾 Step 5: Transform Data to Relational SQLite Database (`Run_05`)
นำเข้าข้อมูลที่ผ่านการตรวจสอบแล้วของ **`ACTIVE_BATCH_ID`** เข้าสู่ฐานข้อมูล SQLite ในตาราง `extracted_documents`, `expense_receipts`, และ `expense_receipt_items`

In [ ]:
print(f"--- [Stage 5] Importing Records into Relational DB for Batch: '{ACTIVE_BATCH_ID}' ---")
if not ACTIVE_BATCH_ID:
    raise ValueError("ACTIVE_BATCH_ID is not set. Please run previous steps first.")

db_result = transform_to_db(batch_id=ACTIVE_BATCH_ID, doc_type=DOC_TYPE, company_code=COMPANY_CODE)
print(f"📊 DB Transformation Summary: {db_result}\n")

# Query database records using Pure SQLAlchemy 2.0
with get_db_session() as session:
    receipt_count = session.scalar(select(func.count()).select_from(ExpenseReceipt))
    item_count = session.scalar(select(func.count()).select_from(ExpenseReceiptItem))
    receipts = session.scalars(select(ExpenseReceipt).limit(10)).all()
    
    print(f"📊 Total Receipts in DB: {receipt_count}")
    print(f"📊 Total Line Items in DB: {item_count}")
    print("\n--- [Database Sample Records] ---")
    for rc in receipts:
        print(f"📄 Receipt ID: {rc.receipt_id} | Merchant: {rc.merchant_name} | Net: {rc.net_amount} THB | Date: {rc.transaction_date}")

## ⚡ Step 6: (Optional) Full Pipeline Single-File Runner
ฟังก์ชันตัวอย่างสำหรับสั่งรัน 1 ไฟล์ตั้งแต่ต้นจนจบ (Stage 1 ➔ Stage 5) โดยส่งผ่าน `batch_id` อย่างเป็นระเบียบ

In [ ]:
def execute_single_file_pipeline(file_path: str, doc_type: str = DOC_TYPE, company_code: str = COMPANY_CODE):
    print(f"🚀 [1/5] Initializing System...")
    init_system(drop_and_recreate=False)
    
    print(f"🚀 [2/5] Splitting and Ingesting: {file_path}...")
    split_res = split_and_match(doc_type=doc_type, input_file=file_path, company_code=company_code)
    if not split_res:
        print("ℹ️ Document held in PENDING or no split output.")
        return False
        
    target_batch = split_res[0]["batch_id"]
    print(f"🚀 [3/5] Extracting with AI [Batch: {target_batch}]...")
    extract_documents(batch_id=target_batch, doc_type=doc_type, company_code=company_code)
    
    print(f"🚀 [4/5] Validating Rules [Batch: {target_batch}]...")
    validate_documents(batch_id=target_batch, doc_type=doc_type, company_code=company_code)
    
    print(f"🚀 [5/5] Transforming to Database [Batch: {target_batch}]...")
    db_res = transform_to_db(batch_id=target_batch, doc_type=doc_type, company_code=company_code)
    print("🎉 Single-File Pipeline Run Completed:", db_res)
    return True

# execute_single_file_pipeline(test_pdf_target)